In [1]:
# Cell 0 — Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, glob, json, warnings
import numpy as np
import pandas as pd
import joblib
from scipy.special import erf
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    matthews_corrcoef)
warnings.filterwarnings('ignore')
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
RUNROOT = os.path.join(DRIVEROOT, RUNID)
OUTPUTDIR = os.path.join(RUNROOT, 'coordinator_v4', 'predictions')
os.makedirs(OUTPUTDIR, exist_ok=True)
print('RUNROOT:', RUNROOT)
print('OUTPUT :', OUTPUTDIR)

Mounted at /content/drive
RUNROOT: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001
OUTPUT : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/coordinator_v4/predictions


In [2]:
# Cell 1 — Discovery (Credit Card only)
MODEL_REGISTRY = {
    'xgb_ae_hybrid':    {'tags': ['xgb_ae_hybrid'],              'cc': ['*xgb*ae*credit*.joblib']},
    'lgbm_classifier':  {'tags': ['lgbm_classifier','memtostrad','memto_strad'], 'cc': ['*lgbm*credit*.joblib','*memto*credit*.joblib']},
    'conv_ae':          {'tags': ['conv_ae','lstm_ae'],           'cc': ['*conv_ae*credit*.joblib','*lstm*credit*.joblib']},
    'ecod':             {'tags': ['ecod'],                        'cc': ['*ecod*credit*.joblib']},
    'isolation_forest': {'tags': ['iforeststrict','iforest_strict'], 'cc': ['*if*credit*.joblib','*iforest*credit*.joblib']},
    'matrix_profile':   {'tags': ['matrix_profile','matrixprofile'], 'cc': ['*mpcreditcard*.joblib','*matrix*credit*.joblib']},
}

def discover():
    found = {}
    for mk, info in MODEL_REGISTRY.items():
        for tag in info['tags']:
            pd_ = os.path.join(RUNROOT, tag, 'predictions')
            if not os.path.isdir(pd_): continue
            for pat in info['cc']:
                for h in glob.glob(os.path.join(pd_, pat)):
                    if mk not in found or os.path.getsize(h) > os.path.getsize(found[mk]):
                        found[mk] = h
    return found

discovered = discover()
print(f'Discovered {len(discovered)} CC bundles:')
for m, p in sorted(discovered.items()):
    print(f'  {m:20s} | {os.path.getsize(p)/1024:8.1f} KB | {os.path.basename(p)}')
cc_models = sorted(discovered.keys())
print(f'\nCC models: {cc_models}')

Discovered 6 CC bundles:
  conv_ae              |   1224.5 KB | conv_ae_creditcard_strict.joblib
  ecod                 |   1224.5 KB | ecod_creditcard_strict.joblib
  isolation_forest     |   1224.5 KB | ifcreditcardstrictpointwise.joblib
  lgbm_classifier      |   1224.8 KB | memtostrad_creditcard.joblib
  matrix_profile       |   1224.5 KB | mpcreditcardstrictpointwise.joblib
  xgb_ae_hybrid        |   1225.0 KB | xgb_ae_hybrid_creditcard_strict.joblib

CC models: ['conv_ae', 'ecod', 'isolation_forest', 'lgbm_classifier', 'matrix_profile', 'xgb_ae_hybrid']


In [3]:
# Cell 2 — Flexible key getter + CC loader
def gv(d, *keys):
    for k in keys:
        if k in d: return d[k]
    for k in keys:
        kn = k.lower().replace('_','')
        for ek, ev in d.items():
            if ek.lower().replace('_','') == kn: return ev
    return None

def load_cc(bundle):
    ents = bundle.get('entities', {})
    ent = ents.get('creditcard') or ents.get('creditcard_test')
    if ent is None and len(ents) == 1: ent = list(ents.values())[0]
    if not ent: raise ValueError(f'No CC entity. Keys={list(ents.keys())}')
    s = gv(ent,'scoresfull','scores_full','scores')
    y = gv(ent,'yfull','y_full','ytrue')
    p = gv(ent,'predfull','pred_full','preds')
    r = gv(ent,'originalrowid','original_row_id')
    t = gv(ent,'threshold')
    return {'scores': np.asarray(s, dtype=np.float64), 'y_true': np.asarray(y, dtype=np.int8),
            'preds': np.asarray(p, dtype=np.int8) if p is not None else None,
            'originalrowid': np.asarray(r, dtype=np.int64) if r is not None else None,
            'threshold': float(t) if t is not None else None}

print('Loaders defined.')

Loaders defined.


In [4]:
# Cell 3 — Load all CC bundles
print('=== LOADING CC BUNDLES ===')
cc_data = {}
for mk in cc_models:
    try:
        d = load_cc(joblib.load(discovered[mk]))
        cc_data[mk] = d
        print(f'  {mk}: {len(d["scores"]):,} samples, {int(d["y_true"].sum())} frauds')
    except Exception as e:
        print(f'  ERROR {mk}: {e}')

print(f'\nLoaded: {sorted(cc_data.keys())}')

=== LOADING CC BUNDLES ===
  conv_ae: 56,962 samples, 98 frauds
  ecod: 56,962 samples, 98 frauds
  isolation_forest: 56,962 samples, 98 frauds
  lgbm_classifier: 56,962 samples, 98 frauds
  matrix_profile: 56,962 samples, 98 frauds
  xgb_ae_hybrid: 56,962 samples, 98 frauds

Loaded: ['conv_ae', 'ecod', 'isolation_forest', 'lgbm_classifier', 'matrix_profile', 'xgb_ae_hybrid']


In [5]:
# Cell 4 — Align CC
ref_mk = list(cc_data.keys())[0]
ref_y = cc_data[ref_mk]['y_true']
ref_rid = cc_data[ref_mk].get('originalrowid')
print(f'Ref: {ref_mk} ({len(ref_y):,} samples, {int(ref_y.sum())} frauds)')
for mk, d in cc_data.items():
    if mk == ref_mk: continue
    if np.array_equal(d['y_true'], ref_y): print(f'  {mk}: OK')
    else:
        rid = d.get('originalrowid')
        if rid is not None and ref_rid is not None and np.array_equal(np.sort(rid), np.sort(ref_rid)):
            inv = np.empty_like(np.argsort(ref_rid)); inv[np.argsort(ref_rid)] = np.arange(len(ref_rid))
            m = inv[np.argsort(rid)]
            for k in ['scores','y_true','preds','originalrowid']:
                if d[k] is not None: d[k] = d[k][m]
            print(f'  {mk}: reordered')
        else: print(f'  WARNING {mk}: mismatch!')
print('Aligned.')

Ref: conv_ae (56,962 samples, 98 frauds)
  ecod: OK
  isolation_forest: OK
  lgbm_classifier: OK
  matrix_profile: OK
  xgb_ae_hybrid: OK
Aligned.


In [6]:
# Cell 5 — Individual CC Metrics
print('='*80)
print('INDIVIDUAL MODEL METRICS — CREDIT CARD')
print('='*80)
cc_metrics = {}
for mk in sorted(cc_data.keys()):
    d = cc_data[mk]
    p = d['preds'] if d['preds'] is not None else (d['scores'] >= (d['threshold'] or np.percentile(d['scores'],99))).astype(np.int8)
    m = {'precision': precision_score(d['y_true'], p, zero_division=0),
         'recall': recall_score(d['y_true'], p, zero_division=0),
         'f1': f1_score(d['y_true'], p, zero_division=0),
         'rocauc': roc_auc_score(d['y_true'], d['scores']) if len(np.unique(d['y_true'])) == 2 else float('nan'),
         'prauc': average_precision_score(d['y_true'], d['scores']) if len(np.unique(d['y_true'])) == 2 else float('nan')}
    cc_metrics[mk] = m
    print(f"  {mk:20s} P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} ROC={m['rocauc']:.4f} PRAUC={m['prauc']:.4f}")
display(pd.DataFrame([{'model': k, **v} for k, v in cc_metrics.items()]).sort_values('f1', ascending=False))

INDIVIDUAL MODEL METRICS — CREDIT CARD
  conv_ae              P=0.7184 R=0.7551 F1=0.7363 ROC=0.9545 PRAUC=0.6879
  ecod                 P=0.3421 R=0.5306 F1=0.4160 ROC=0.9616 PRAUC=0.3536
  isolation_forest     P=0.4640 R=0.5918 F1=0.5202 ROC=0.9596 PRAUC=0.5065
  lgbm_classifier      P=0.9302 R=0.8163 F1=0.8696 ROC=0.9372 PRAUC=0.8480
  matrix_profile       P=0.8211 R=0.7959 F1=0.8083 ROC=0.9646 PRAUC=0.6957
  xgb_ae_hybrid        P=0.9412 R=0.8163 F1=0.8743 ROC=0.9769 PRAUC=0.8745


,model,precision,recall,f1,rocauc,prauc
5,xgb_ae_hybrid,0.941176,0.816327,0.874317,0.976949,0.874524
3,lgbm_classifier,0.930233,0.816327,0.869565,0.937231,0.848038
4,matrix_profile,0.821053,0.795918,0.808290,0.964591,0.695749
0,conv_ae,0.718447,0.755102,0.736318,0.954514,0.687856
2,isolation_forest,0.464000,0.591837,0.520179,0.959610,0.506539
1,ecod,0.342105,0.530612,0.416000,0.961591,0.353613


In [7]:
# Cell 6 — CC Fusion with TYPE-AWARE score normalization
# Research brief hierarchy:
#   - XGBoost/LightGBM probabilities -> use directly (already calibrated [0,1])
#   - IF/ECOD anomaly scores -> Gaussian scaling via erf((S-mu)/(sigma*sqrt(2)))
#   - Conv-AE reconstruction error / Matrix Profile -> rank normalization (heavy right tails)

SCORE_TYPE = {
    'xgb_ae_hybrid':    'probability',   # XGBoost predict_proba output
    'lgbm_classifier':  'probability',   # LightGBM predict_proba output
    'isolation_forest': 'gaussian',      # negative decision_function
    'ecod':             'gaussian',      # ECOD outlier scores
    'conv_ae':          'rank',          # AE reconstruction error (heavy tail)
    'matrix_profile':   'rank',          # Mahalanobis distance (heavy tail)
}

def normalize_score(scores, method):
    """Type-aware score normalization."""
    if method == 'probability':
        # Already [0,1] calibrated — clip for safety
        return np.clip(scores, 0.0, 1.0)
    elif method == 'gaussian':
        # Gaussian CDF scaling: maps to [0,1] assuming roughly normal raw scores
        mu = np.mean(scores)
        sigma = np.std(scores) + 1e-12
        return 0.5 * (1.0 + erf((scores - mu) / (sigma * np.sqrt(2.0))))
    elif method == 'rank':
        # Rank-based normalization: robust to heavy tails
        return rankdata(scores, method='average') / len(scores)
    else:
        # Fallback: MinMax
        smin, smax = scores.min(), scores.max()
        return (scores - smin) / (smax - smin + 1e-12)

model_keys_cc = sorted(cc_data.keys())
y_cc = ref_y.copy()

# Build BOTH raw and normalized score matrices
raw_scores = np.column_stack([cc_data[m]['scores'] for m in model_keys_cc])
norm_scores = np.column_stack([
    normalize_score(cc_data[m]['scores'], SCORE_TYPE.get(m, 'minmax'))
    for m in model_keys_cc
])

print('='*80)
print('SCORE NORMALIZATION DIAGNOSTICS')
print('='*80)
for i, m in enumerate(model_keys_cc):
    stype = SCORE_TYPE.get(m, 'minmax')
    raw_col = raw_scores[:, i]
    norm_col = norm_scores[:, i]
    print(f'  {m:20s} type={stype:12s} raw=[{raw_col.min():.4f}, {raw_col.max():.4f}] '
          f'norm=[{norm_col.min():.4f}, {norm_col.max():.4f}]')

# --- LR stacking with TYPE-AWARE normalization (no MinMaxScaler) ---
def lr_stack_v4(X, y, names, C=1.0, label='', calibrate=False):
    """OOF LR stacking on pre-normalized scores (no additional scaling)."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    probs = np.zeros(len(y), dtype=np.float64)
    for tr, te in cv.split(X, y):
        lr = LogisticRegression(class_weight='balanced', max_iter=2000,
                                random_state=RANDOM_STATE, C=C)
        if calibrate:
            # Wrap LR in isotonic calibration
            base_lr = LogisticRegression(class_weight='balanced', max_iter=2000,
                                         random_state=RANDOM_STATE, C=C)
            lr = CalibratedClassifierCV(base_lr, method='isotonic', cv=3)
        lr.fit(X[tr], y[tr])
        probs[te] = lr.predict_proba(X[te])[:, 1]
    pr, rc, th = precision_recall_curve(y, probs)
    f1s = 2*pr[:-1]*rc[:-1]/(pr[:-1]+rc[:-1]+1e-12)
    bi = int(np.nanargmax(f1s)); bt = float(th[bi])
    pds = (probs >= bt).astype(np.int8)
    m = {'precision': precision_score(y, pds, zero_division=0),
         'recall': recall_score(y, pds, zero_division=0),
         'f1': f1_score(y, pds, zero_division=0),
         'rocauc': roc_auc_score(y, probs),
         'prauc': average_precision_score(y, probs)}
    cal_tag = '+isotonic' if calibrate else ''
    print(f'  {label}{cal_tag}: P={m["precision"]:.4f} R={m["recall"]:.4f} '
          f'F1={m["f1"]:.4f} ROC={m["rocauc"]:.4f} PRAUC={m["prauc"]:.4f}')
    return m, probs, pds, bt

def wavg_v4(X, y, names, f1s_list, label=''):
    """F1^2-weighted average on pre-normalized scores."""
    w = np.array([f**2 for f in f1s_list]); w = w/w.sum()
    ws = (X * w[None, :]).sum(axis=1)
    pr, rc, th = precision_recall_curve(y, ws)
    f1s = 2*pr[:-1]*rc[:-1]/(pr[:-1]+rc[:-1]+1e-12)
    bi = int(np.nanargmax(f1s)); bt = float(th[bi])
    pds = (ws >= bt).astype(np.int8)
    m = {'precision': precision_score(y, pds, zero_division=0),
         'recall': recall_score(y, pds, zero_division=0),
         'f1': f1_score(y, pds, zero_division=0),
         'rocauc': roc_auc_score(y, ws),
         'prauc': average_precision_score(y, ws)}
    print(f'  {label}: P={m["precision"]:.4f} R={m["recall"]:.4f} '
          f'F1={m["f1"]:.4f} ROC={m["rocauc"]:.4f} W={dict(zip(names,w.round(3)))}')
    return m, ws, pds, bt

print('\n' + '='*80)
print('CC FUSION EXPERIMENTS (v4 — type-aware normalization)')
print('='*80)

# Top-K selection (F1 >= 0.75)
tk_mask = [cc_metrics[m]['f1'] >= 0.75 for m in model_keys_cc]
tk_names = [m for m, k in zip(model_keys_cc, tk_mask) if k]
tk_norm = norm_scores[:, tk_mask]
tk_raw = raw_scores[:, tk_mask]
print(f'\nTop-K models (F1>=0.75): {tk_names}')

# --- v3.3 strategies (MinMaxScaler baseline for comparison) ---
print('\n--- v3.3 BASELINE (MinMaxScaler) ---')
sc_mm = MinMaxScaler()
tk_mm = sc_mm.fit_transform(tk_raw)
w_tk_f1 = [cc_metrics[m]['f1'] for m in tk_names]
w_old = np.array([f**2 for f in w_tk_f1]); w_old = w_old/w_old.sum()
ws_old = (tk_mm * w_old[None,:]).sum(axis=1)
pr_o, rc_o, th_o = precision_recall_curve(y_cc, ws_old)
f1s_o = 2*pr_o[:-1]*rc_o[:-1]/(pr_o[:-1]+rc_o[:-1]+1e-12)
bi_o = int(np.nanargmax(f1s_o)); bt_o = float(th_o[bi_o])
pds_o = (ws_old >= bt_o).astype(np.int8)
m_old = {'precision': precision_score(y_cc, pds_o, zero_division=0),
         'recall': recall_score(y_cc, pds_o, zero_division=0),
         'f1': f1_score(y_cc, pds_o, zero_division=0),
         'rocauc': roc_auc_score(y_cc, ws_old),
         'prauc': average_precision_score(y_cc, ws_old)}
print(f'  WAvg-topK-v3.3: P={m_old["precision"]:.4f} R={m_old["recall"]:.4f} '
      f'F1={m_old["f1"]:.4f} ROC={m_old["rocauc"]:.4f} (MinMax baseline)')

# --- v4 strategies (type-aware normalization) ---
print('\n--- v4 TYPE-AWARE NORMALIZATION ---')

# LR stacking variants
m1, p1, pd1, t1 = lr_stack_v4(norm_scores, y_cc, model_keys_cc, C=1.0, label='LR-all-C1')
m2, p2, pd2, t2 = lr_stack_v4(tk_norm, y_cc, tk_names, C=1.0, label='LR-topK-C1')
m3, p3, pd3, t3 = lr_stack_v4(tk_norm, y_cc, tk_names, C=0.5, label='LR-topK-C0.5')

# LR with isotonic calibration
m4, p4, pd4, t4 = lr_stack_v4(norm_scores, y_cc, model_keys_cc, C=1.0, label='LR-all-C1', calibrate=True)
m5, p5, pd5, t5 = lr_stack_v4(tk_norm, y_cc, tk_names, C=1.0, label='LR-topK-C1', calibrate=True)

# Weighted average on normalized scores
all_f1 = [cc_metrics[m]['f1'] for m in model_keys_cc]
m6, w6, pd6, t6 = wavg_v4(norm_scores, y_cc, model_keys_cc, all_f1, 'WAvg-all-v4')
tk_f1 = [cc_metrics[m]['f1'] for m in tk_names]
m7, w7, pd7, t7 = wavg_v4(tk_norm, y_cc, tk_names, tk_f1, 'WAvg-topK-v4')

# XGB+LGBM pair (on normalized)
pair_names = [m for m in model_keys_cc if m in ('xgb_ae_hybrid','lgbm_classifier')]
if len(pair_names) == 2:
    pair_mask = [m in pair_names for m in model_keys_cc]
    pair_norm = norm_scores[:, pair_mask]
    m8, p8, pd8, t8 = lr_stack_v4(pair_norm, y_cc, pair_names, C=1.0, label='LR-XGB+LGBM')
else:
    m8 = {'f1': 0}; p8 = p1; pd8 = pd1; t8 = t1

all_m = [
    ('LR_all_v4', m1, p1, pd1, t1),
    ('LR_topK_v4', m2, p2, pd2, t2),
    ('LR_topK_C05_v4', m3, p3, pd3, t3),
    ('LR_all_isotonic', m4, p4, pd4, t4),
    ('LR_topK_isotonic', m5, p5, pd5, t5),
    ('WAvg_all_v4', m6, w6, pd6, t6),
    ('WAvg_topK_v4', m7, w7, pd7, t7),
    ('LR_XGB_LGBM_v4', m8, p8, pd8, t8),
    ('WAvg_topK_v3.3', m_old, ws_old, pds_o, bt_o),  # baseline
]
best = max(all_m, key=lambda x: x[1]['f1'])
print(f'\n>>> Best CC fusion: {best[0]} F1={best[1]["f1"]:.4f}')
best_cc_name, best_cc_m, best_cc_probs, best_cc_preds, best_cc_thr = best

# Show improvement over v3.3
delta = best[1]['f1'] - m_old['f1']
print(f'    vs v3.3 WAvg_topK: {"+" if delta >= 0 else ""}{delta:.4f}')

SCORE NORMALIZATION DIAGNOSTICS
  conv_ae              type=rank         raw=[0.1114, 8.5330] norm=[0.0000, 1.0000]
  ecod                 type=gaussian     raw=[42.3348, 293.5758] norm=[0.0472, 1.0000]
  isolation_forest     type=gaussian     raw=[-0.2797, 0.2074] norm=[0.2013, 1.0000]
  lgbm_classifier      type=probability  raw=[0.0000, 1.0000] norm=[0.0000, 1.0000]
  matrix_profile       type=rank         raw=[2.2483, 1450.6675] norm=[0.0000, 1.0000]
  xgb_ae_hybrid        type=probability  raw=[0.0000, 1.0000] norm=[0.0000, 1.0000]

CC FUSION EXPERIMENTS (v4 — type-aware normalization)

Top-K models (F1>=0.75): ['lgbm_classifier', 'matrix_profile', 'xgb_ae_hybrid']

--- v3.3 BASELINE (MinMaxScaler) ---
  WAvg-topK-v3.3: P=0.9130 R=0.8571 F1=0.8842 ROC=0.9694 (MinMax baseline)

--- v4 TYPE-AWARE NORMALIZATION ---
  LR-all-C1: P=0.9130 R=0.8571 F1=0.8842 ROC=0.9585 PRAUC=0.8555
  LR-topK-C1: P=0.9130 R=0.8571 F1=0.8842 ROC=0.9581 PRAUC=0.8592
  LR-topK-C0.5: P=0.9130 R=0.8571 F1=0.8

In [8]:
# Cell 8 — DIVERSITY ANALYSIS (Spearman rho + Yule's Q)
# This proves the ensemble composition is justified

print('='*80)
print('ENSEMBLE DIVERSITY ANALYSIS — CREDIT CARD')
print('='*80)

# --- Spearman rank correlation on anomaly scores ---
print('\n--- Spearman rho (anomaly score correlation) ---')
print('Expected: rho(XGB,LGBM) ~ 0.85-0.92, rho(ECOD,trees) ~ 0.4-0.6,')
print('          rho(AE,trees) ~ 0.1-0.3, rho(MP,trees) ~ 0.0-0.2')
print()

spearman_matrix = np.zeros((len(model_keys_cc), len(model_keys_cc)))
for i, m1 in enumerate(model_keys_cc):
    for j, m2 in enumerate(model_keys_cc):
        if i <= j:
            rho, _ = spearmanr(cc_data[m1]['scores'], cc_data[m2]['scores'])
            spearman_matrix[i, j] = rho
            spearman_matrix[j, i] = rho

rho_df = pd.DataFrame(spearman_matrix, index=model_keys_cc, columns=model_keys_cc)
print('Spearman rho matrix:')
display(rho_df.round(4))

# --- Yule's Q on binarized predictions ---
print('\n--- Yule\'s Q (prediction agreement) ---')
print('Q near 0 = independent errors (ideal for ensemble)')
print('Q near 1 = redundant models (no diversity)')
print()

all_preds = {}
for mk in model_keys_cc:
    d = cc_data[mk]
    p = d['preds'] if d['preds'] is not None else \
        (d['scores'] >= (d['threshold'] or np.percentile(d['scores'], 99))).astype(np.int8)
    all_preds[mk] = p

def yules_q(p1, p2):
    """Yule's Q statistic: measures association between two binary classifiers."""
    a = int(((p1 == 1) & (p2 == 1)).sum())  # both positive
    b = int(((p1 == 1) & (p2 == 0)).sum())  # p1 pos, p2 neg
    c = int(((p1 == 0) & (p2 == 1)).sum())  # p1 neg, p2 pos
    d = int(((p1 == 0) & (p2 == 0)).sum())  # both negative
    denom = a*d + b*c
    if denom == 0: return 0.0
    return (a*d - b*c) / denom

q_matrix = np.zeros((len(model_keys_cc), len(model_keys_cc)))
for i, m1 in enumerate(model_keys_cc):
    for j, m2 in enumerate(model_keys_cc):
        if i == j:
            q_matrix[i, j] = 1.0
        elif i < j:
            q = yules_q(all_preds[m1], all_preds[m2])
            q_matrix[i, j] = q
            q_matrix[j, i] = q

q_df = pd.DataFrame(q_matrix, index=model_keys_cc, columns=model_keys_cc)
print("Yule's Q matrix:")
display(q_df.round(4))

# Summary statistics
print('\n--- Diversity Summary ---')
# Get off-diagonal elements
rho_vals = [spearman_matrix[i,j] for i in range(len(model_keys_cc))
            for j in range(i+1, len(model_keys_cc))]
q_vals = [q_matrix[i,j] for i in range(len(model_keys_cc))
          for j in range(i+1, len(model_keys_cc))]
print(f'Mean pairwise Spearman rho: {np.mean(rho_vals):.4f} (lower = more diverse)')
print(f'Mean pairwise Yule\'s Q:    {np.mean(q_vals):.4f} (lower = more diverse)')

# Highlight top-K diversity specifically
tk_indices = [model_keys_cc.index(m) for m in tk_names]
tk_rho = [spearman_matrix[i,j] for i_idx, i in enumerate(tk_indices)
          for j in tk_indices[i_idx+1:]]
tk_q = [q_matrix[i,j] for i_idx, i in enumerate(tk_indices)
        for j in tk_indices[i_idx+1:]]
print(f'\nTop-K ({tk_names}) pairwise:')
print(f'  Mean Spearman rho: {np.mean(tk_rho):.4f}')
print(f'  Mean Yule\'s Q:    {np.mean(tk_q):.4f}')

ENSEMBLE DIVERSITY ANALYSIS — CREDIT CARD

--- Spearman rho (anomaly score correlation) ---
Expected: rho(XGB,LGBM) ~ 0.85-0.92, rho(ECOD,trees) ~ 0.4-0.6,
          rho(AE,trees) ~ 0.1-0.3, rho(MP,trees) ~ 0.0-0.2

Spearman rho matrix:


,conv_ae,ecod,isolation_forest,lgbm_classifier,matrix_profile,xgb_ae_hybrid
conv_ae,1.0000,0.7079,0.7044,-0.0559,0.3585,0.3496
ecod,0.7079,1.0000,0.9108,0.0249,0.4528,0.2977
isolation_forest,0.7044,0.9108,1.0000,0.0627,0.4615,0.3627
lgbm_classifier,-0.0559,0.0249,0.0627,1.0000,-0.0092,0.1838
matrix_profile,0.3585,0.4528,0.4615,-0.0092,1.0000,0.1546
xgb_ae_hybrid,0.3496,0.2977,0.3627,0.1838,0.1546,1.0000



--- Yule's Q (prediction agreement) ---
Q near 0 = independent errors (ideal for ensemble)
Q near 1 = redundant models (no diversity)

Yule's Q matrix:


,conv_ae,ecod,isolation_forest,lgbm_classifier,matrix_profile,xgb_ae_hybrid
conv_ae,1.0000,0.9987,0.9994,0.9998,0.9999,0.9999
ecod,0.9987,1.0000,0.9999,0.9978,0.9981,0.9979
isolation_forest,0.9994,0.9999,1.0000,0.9989,0.9991,0.9990
lgbm_classifier,0.9998,0.9978,0.9989,1.0000,0.9999,1.0000
matrix_profile,0.9999,0.9981,0.9991,0.9999,1.0000,0.9999
xgb_ae_hybrid,0.9999,0.9979,0.9990,1.0000,0.9999,1.0000



--- Diversity Summary ---
Mean pairwise Spearman rho: 0.3311 (lower = more diverse)
Mean pairwise Yule's Q:    0.9992 (lower = more diverse)

Top-K (['lgbm_classifier', 'matrix_profile', 'xgb_ae_hybrid']) pairwise:
  Mean Spearman rho: 0.1097
  Mean Yule's Q:    0.9999


In [9]:
# Cell 9 — Bootstrap Confidence Intervals for Ensemble F1
print('='*80)
print('BOOTSTRAP CONFIDENCE INTERVALS (1000 resamples)')
print('='*80)

N_BOOT = 1000
np.random.seed(RANDOM_STATE)

def bootstrap_f1(y_true, y_pred, n_boot=1000):
    """Bootstrap F1 score with 95% confidence interval."""
    n = len(y_true)
    f1s = np.zeros(n_boot)
    for b in range(n_boot):
        idx = np.random.randint(0, n, size=n)
        f1s[b] = f1_score(y_true[idx], y_pred[idx], zero_division=0)
    return np.percentile(f1s, [2.5, 50, 97.5])

# Ensemble CI
ens_ci = bootstrap_f1(y_cc, best_cc_preds, N_BOOT)
print(f'\nBest Ensemble ({best_cc_name}):')
print(f'  F1 = {ens_ci[1]:.4f}  95% CI [{ens_ci[0]:.4f}, {ens_ci[2]:.4f}]')

# Individual model CIs for comparison
print('\nIndividual models:')
for mk in sorted(cc_data.keys(), key=lambda k: cc_metrics[k]['f1'], reverse=True):
    d = cc_data[mk]
    p = d['preds'] if d['preds'] is not None else \
        (d['scores'] >= (d['threshold'] or np.percentile(d['scores'], 99))).astype(np.int8)
    ci = bootstrap_f1(d['y_true'], p, N_BOOT)
    print(f'  {mk:20s}: F1 = {ci[1]:.4f}  95% CI [{ci[0]:.4f}, {ci[2]:.4f}]')

# v3.3 baseline CI
v33_ci = bootstrap_f1(y_cc, pds_o, N_BOOT)
print(f'\nv3.3 WAvg_topK:')
print(f'  F1 = {v33_ci[1]:.4f}  95% CI [{v33_ci[0]:.4f}, {v33_ci[2]:.4f}]')

# MCC for the ensemble (more robust on imbalanced data)
mcc_ens = matthews_corrcoef(y_cc, best_cc_preds)
print(f'\nEnsemble MCC: {mcc_ens:.4f}')

BOOTSTRAP CONFIDENCE INTERVALS (1000 resamples)

Best Ensemble (LR_topK_isotonic):
  F1 = 0.8901  95% CI [0.8387, 0.9341]

Individual models:
  xgb_ae_hybrid       : F1 = 0.8757  95% CI [0.8229, 0.9231]
  lgbm_classifier     : F1 = 0.8700  95% CI [0.8148, 0.9195]
  matrix_profile      : F1 = 0.8068  95% CI [0.7472, 0.8634]
  conv_ae             : F1 = 0.7358  95% CI [0.6667, 0.8000]
  isolation_forest    : F1 = 0.5221  95% CI [0.4348, 0.5984]
  ecod                : F1 = 0.4135  95% CI [0.3360, 0.4923]

v3.3 WAvg_topK:
  F1 = 0.8857  95% CI [0.8306, 0.9302]

Ensemble MCC: 0.8893


In [10]:
# Cell 10 — Final Comparison
print('='*80)
print('FINAL COMPARISON — COORDINATOR v4 (Credit Card)')
print('='*80)

rows = []
for mk, m in cc_metrics.items():
    rows.append({'method': mk, 'cc_f1': m['f1'], 'cc_rocauc': m['rocauc'], 'cc_prauc': m['prauc']})
rows.append({'method': f'ENSEMBLE_{best_cc_name}', 'cc_f1': best_cc_m['f1'],
             'cc_rocauc': best_cc_m['rocauc'], 'cc_prauc': best_cc_m['prauc']})
rows.append({'method': 'ENSEMBLE_WAvg_topK_v3.3', 'cc_f1': m_old['f1'],
             'cc_rocauc': m_old['rocauc'], 'cc_prauc': m_old['prauc']})
print('\nCredit Card:')
display(pd.DataFrame(rows).sort_values('cc_f1', ascending=False))

bi_f1 = max(m['f1'] for m in cc_metrics.values())
bi_n = max(cc_metrics, key=lambda k: cc_metrics[k]['f1'])
print(f'Best individual: {bi_n} F1={bi_f1:.4f}')
print(f'Best ensemble  : {best_cc_name} F1={best_cc_m["f1"]:.4f}')
if best_cc_m['f1'] > bi_f1:
    print(f'>>> ENSEMBLE WINS by {best_cc_m["f1"]-bi_f1:.4f}')
else:
    print(f'>>> Gap: {bi_f1-best_cc_m["f1"]:.4f}')

# --- Agreement analysis ---
print('\n' + '='*80)
print('ENSEMBLE DECISION EXPLAINABILITY')
print('='*80)
agreement = np.zeros(len(y_cc), dtype=int)
for mk in model_keys_cc:
    agreement += all_preds[mk]
print(f'Full agreement (all {len(model_keys_cc)} say fraud): {(agreement == len(model_keys_cc)).sum()}')
print(f'Majority agree (>50%): {(agreement > len(model_keys_cc)/2).sum()}')
print(f'No model flags: {(agreement == 0).sum()}')
fraud_idx = np.where(y_cc == 1)[0]
print(f'\nFor {len(fraud_idx)} actual frauds:')
for n_agree in range(len(model_keys_cc)+1):
    caught = ((agreement[fraud_idx] == n_agree)).sum()
    if caught > 0:
        print(f'  Caught by exactly {n_agree}/{len(model_keys_cc)} models: {caught} frauds')

FINAL COMPARISON — COORDINATOR v4 (Credit Card)

Credit Card:


,method,cc_f1,cc_rocauc,cc_prauc
6,ENSEMBLE_LR_topK_isotonic,0.888889,0.957815,0.847858
7,ENSEMBLE_WAvg_topK_v3.3,0.884211,0.969443,0.852838
5,xgb_ae_hybrid,0.874317,0.976949,0.874524
3,lgbm_classifier,0.869565,0.937231,0.848038
4,matrix_profile,0.808290,0.964591,0.695749
0,conv_ae,0.736318,0.954514,0.687856
2,isolation_forest,0.520179,0.959610,0.506539
1,ecod,0.416000,0.961591,0.353613


Best individual: xgb_ae_hybrid F1=0.8743
Best ensemble  : LR_topK_isotonic F1=0.8889
>>> ENSEMBLE WINS by 0.0146

ENSEMBLE DECISION EXPLAINABILITY
Full agreement (all 6 say fraud): 48
Majority agree (>50%): 83
No model flags: 56759

For 98 actual frauds:
  Caught by exactly 0/6 models: 13 frauds
  Caught by exactly 1/6 models: 2 frauds
  Caught by exactly 2/6 models: 4 frauds
  Caught by exactly 3/6 models: 6 frauds
  Caught by exactly 4/6 models: 18 frauds
  Caught by exactly 5/6 models: 8 frauds
  Caught by exactly 6/6 models: 47 frauds


In [11]:
# Cell 11 — Export
cc_ens = {
    'dataset': 'creditcard',
    'model': f'coordinator_v4_{best_cc_name}',
    'base_models': model_keys_cc,
    'fusion_strategy': best_cc_name,
    'score_normalization': SCORE_TYPE,
    'entities': {'creditcard': {
        'entityid': 'creditcard',
        'scoresfull': best_cc_probs.astype(np.float32),
        'yfull': y_cc,
        'predfull': best_cc_preds,
        'originalrowid': ref_rid,
        'threshold': best_cc_thr}},
    'metrics': best_cc_m,
    'individual_metrics': cc_metrics,
    'model_weights': {m: float(cc_metrics[m]['f1']**2) for m in model_keys_cc},
    'diversity': {
        'spearman_rho_mean': float(np.mean(rho_vals)),
        'yules_q_mean': float(np.mean(q_vals)),
    },
}
joblib.dump(cc_ens, os.path.join(OUTPUTDIR, 'coordinator_v4_creditcard.joblib'))
print('Saved CC ensemble')

rows = [{'dataset': 'creditcard', 'method': best_cc_name, **best_cc_m, 'n_models': len(model_keys_cc)}]
s = pd.DataFrame(rows)
s.to_csv(os.path.join(OUTPUTDIR, 'coordinator_v4_summary.csv'), index=False)
print('Saved summary')
display(s)

with open(os.path.join(OUTPUTDIR, 'coordinator_v4_manifest.json'), 'w') as f:
    json.dump({
        'runid': RUNID, 'version': 'v4.0',
        'best_cc': best_cc_name,
        'cc_models': model_keys_cc,
        'score_normalization': SCORE_TYPE,
        'cc_weights': {m: round(cc_metrics[m]['f1']**2, 6) for m in model_keys_cc},
        'diversity_spearman_mean': float(np.mean(rho_vals)),
        'diversity_yules_q_mean': float(np.mean(q_vals)),
        'outputdir': OUTPUTDIR,
    }, f, indent=2)

print(f'\nCoordinator v4.0 complete (Credit Card only).')
print(f'Outputs: {OUTPUTDIR}')

Saved CC ensemble
Saved summary


,dataset,method,precision,recall,f1,rocauc,prauc,n_models
0,creditcard,LR_topK_isotonic,0.923077,0.857143,0.888889,0.957815,0.847858,6



Coordinator v4.0 complete (Credit Card only).
Outputs: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/coordinator_v4/predictions
